# Tempo de processamento

In [1]:
import sys
from pathlib import Path

# O kernel inicia na pasta DESTE caderno — localiza a raiz do projeto:
RAIZ = Path.cwd()
if not (RAIZ / "src" / "leitura.py").exists():
    RAIZ = RAIZ.parent          # kernel começou dentro de notebooks/
sys.path.insert(0, str(RAIZ))   # raiz no path -> 'from src...' funciona

# Imports padrão de todos os cadernos:
from src.leitura import (PASTA_IMAGENS, PASTA_RESULTADOS, carregar_imagem,
                         nome_da_mascara, criar_pastas_resultados, auditar_base)
import cv2
import numpy as np
import matplotlib.pyplot as plt

print("raiz do projeto:", RAIZ)

raiz do projeto: /home/isadora/Documentos/ProcessIMg


In [2]:
import time
import cv2

from src.leitura import carregar_imagem, nome_da_mascara
from src.preprocessamento import para_cinza, para_hsv
from src.segmentacao import binarizar, otsu
from src.morfologia import pos_processar
from src.avaliacao import iou, dice, medir
from src.filtragem import filtrar_mediana

In [3]:
for nome in ["ISIC_0010206.jpg", "ISIC_0012940.jpg"]:   # 1 MP × 10 MP
    nome_curto = Path(nome).stem
    tempos = {}
    img, tempos["leitura"] = medir(carregar_imagem, PASTA_IMAGENS / nome)
    cinza, tempos["cinza"] = medir(para_cinza, img)
    s, tempos["S (hsv)"] = medir(lambda: para_hsv(img)[:, :, 1])
    cinza_f, tempos["mediana k=5"] = medir(filtrar_mediana, cinza, 5)
    t_otsu, tempos["otsu"] = medir(otsu, cinza_f)
    bruta, tempos["binarizar"] = medir(binarizar, cinza_f, t_otsu, True)
    final, tempos["pos_processar"] = medir(pos_processar, bruta)
    ref = cv2.imread(str(nome_da_mascara(PASTA_IMAGENS / nome)), cv2.IMREAD_GRAYSCALE)
    v_iou, tempos["iou"] = medir(iou, final, ref)
    v_dice, tempos["dice"] = medir(dice, final, ref)
    total = sum(tempos.values())

    print(f"--- {nome_curto} ({img.shape[1]}×{img.shape[0]}) | IoU={v_iou:.3f} Dice={v_dice:.3f} ---")
    for etapa, ms in tempos.items():
        print(f"  {etapa:<14} {ms:>9.1f} ms")
    print(f"  {'TOTAL':<14} {total:>9.1f} ms\n")

--- ISIC_0010206 (1024×768) | IoU=0.766 Dice=0.867 ---
  leitura              9.8 ms
  cinza                0.3 ms
  S (hsv)              0.9 ms
  mediana k=5          1.8 ms
  otsu                 3.6 ms
  binarizar            3.0 ms
  pos_processar        6.7 ms
  iou                  1.1 ms
  dice                 1.1 ms
  TOTAL               28.4 ms

--- ISIC_0012940 (3872×2592) | IoU=0.052 Dice=0.099 ---
  leitura            105.5 ms
  cinza               17.0 ms
  S (hsv)             36.8 ms
  mediana k=5          8.8 ms
  otsu               106.3 ms
  binarizar          144.1 ms
  pos_processar      139.0 ms
  iou                 13.4 ms
  dice                13.8 ms
  TOTAL              584.7 ms

